# 22 · Model tiering economics

## Goal

Re-pin extraction to a cheap model, routing to a mid model, and drafting to
a frontier model — one connected agent per tier — and measure the
cost/quality Pareto on the shared eval set. Wire prompt caching where it
actually pays off.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.admin import prerequisite_report
known = {
    "external_models_ppac": None, "external_models_provider": None,
    "preview_experimental_models": None, "move_data_across_regions": None,
}
print(prerequisite_report(known))
print("If any external-model row above isn't OK, stop here — this notebook needs at least one non-Microsoft model tier to demonstrate the Pareto tradeoff meaningfully.")


## Concept

**Finding #4 is why this notebook exists in this shape.** Model is selected
per agent, not per step — there's no way to say "use a cheap model for this
one turn" inside a single agent's config. Cost tiering therefore happens
via connected agents: an extraction agent pinned to a cheap model, a
routing agent pinned to a mid model, and the drafting specialist from `20`
re-pinned to a frontier model for the highest-judgement task. Three agents,
three `copilot.yaml`s, three `model:` blocks — that's the whole mechanism.

**Finding #5, made concrete:** if any tier here uses a non-Microsoft model
(Anthropic, Mistral, xAI), it needs the PPAC environment/group switch *and*
the separate per-provider M365 admin center approval — two independent
gates, easy to conflate when debugging why a tier "isn't available." Preview
or experimental models need a third and fourth switch respectively. The
prereqs cell above is there so a missing switch shows as a clear message
here, not a mysterious empty response mid-Pareto-run.


## Build


### Three agents, three tiers


In [ ]:
from pathlib import Path
from csx.pac import copilot_init, copilot_push
import yaml, subprocess

tiers = {
    "extraction-agent": {"provider": "microsoft", "name": "gpt-4o-mini", "task": "Extract structured fields from renewal notices. No judgement calls."},
    "routing-agent": {"provider": "microsoft", "name": "gpt-5-mini", "task": "Decide routine vs escalation vs human-review from spend/performance signals."},
    # frontier tier re-pins the existing drafting-specialist from 20 — no new agent needed
}

for agent_id, cfg in tiers.items():
    ws = Path(f"../agents/{agent_id}")
    copilot_init(ws)
    copilot_yaml = yaml.safe_load((ws / "copilot.yaml").read_text()) if (ws / "copilot.yaml").exists() else {}
    copilot_yaml["model"] = {"provider": cfg["provider"], "name": cfg["name"]}
    (ws / "copilot.yaml").write_text(yaml.dump(copilot_yaml, sort_keys=False))
    (ws / "instructions.md").write_text(f"# {agent_id}\n\n{cfg['task']}\n")
    copilot_push(ws)
    subprocess.run(["pac", "copilot", "publish", "--name", f"crd_{agent_id}"], check=True)

# Re-pin drafting-specialist to a frontier model for the highest-judgement task
drafting = Path("../agents/drafting-specialist")
dyaml = yaml.safe_load((drafting / "copilot.yaml").read_text())
dyaml["model"] = {"provider": "anthropic", "name": "claude-opus-5"}  # requires both external-model switches above
(drafting / "copilot.yaml").write_text(yaml.dump(dyaml, sort_keys=False))
copilot_push(drafting)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_drafting-specialist"], check=True)


### Wire both new tiers into the spine agent's connected agents


In [ ]:
spine = Path("../agents/contract-renewal-desk")
copilot_yaml = yaml.safe_load((spine / "copilot.yaml").read_text())
copilot_yaml["connectedAgents"] += [
    {"id": "extraction-agent", "schemaName": "crd_extraction-agent", "description": "Extract structured fields (supplier, date, uplift) from free text before any other step."},
    {"id": "routing-agent", "schemaName": "crd_routing-agent", "description": "Decide renewal routing (routine/escalation/human-review) from spend/performance signals."},
]
(spine / "copilot.yaml").write_text(yaml.dump(copilot_yaml, sort_keys=False))
from csx.pac import copilot_push
copilot_push(spine)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


### Prompt caching on the frontier tier — where it actually pays off


In [ ]:
# Prompt caching helps most on the frontier (highest per-token cost) tier,
# and only for the stable prefix — the negotiation playbook and the
# specialist's own instructions, not the per-call supplier data. Confirm
# the specialist's prompt structure puts stable content first.
instructions_text = (drafting / "instructions.md").read_text()
print(f"stable prefix ({len(instructions_text)} chars) — cache this; supplier-specific summary appended per call, not cached")


## Verify

Same harness, same golden set, every notebook.


The Pareto measurement: same cases, three tiers, cost vs pass rate.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

extraction_suite = run_suite(client, cases=load_golden(tags=["extraction"]), credit_meter=meter, min_pass_rate=0.8)
routing_suite = run_suite(client, cases=load_golden(tags=["routing"]), credit_meter=meter, min_pass_rate=0.8)
drafting_suite = run_suite(client, cases=load_golden(tags=["multi-agent"]), credit_meter=meter, min_pass_rate=0.8)

for name, suite in [("extraction (cheap)", extraction_suite), ("routing (mid)", routing_suite), ("drafting (frontier)", drafting_suite)]:
    print(f"{name:22} pass_rate={suite.pass_rate:.0%}  credits={suite.total_credits:.2f}  p50={suite.p50_latency_ms:.0f}ms")


## Cost


In [ ]:
total = extraction_suite.total_credits + routing_suite.total_credits + drafting_suite.total_credits
meter.report_cost("22", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=total,
                   note="3-tier Pareto run — compare per-tier credits/pass_rate above before deciding tiering is worth the added complexity")


## Teardown


In [ ]:
print("No teardown — the 3-tier connected-agent structure is the multi-agent shape carried through 23-25.")
